In [1]:
import pandas as pd
import numpy as np
from scapy.all import PacketList,rdpcap, TCP,IP,UDP
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta
import os
import matplotlib.pyplot as plt
import pickle
import torch

In [ ]:
# Read all the pcap files available along with the mac address of the corresponding device
packets = PacketList()
pathlist = Path('captures_IoT-Sentinel').glob('**/Setup-C-*.pcap')
for path in pathlist:
    # because path is object not string
    path_in_str = str(path)
    #print("Name of the device : " + path_in_str.split('/')[-2])
    parent_path = os.path.dirname(path_in_str)   
    mac = open(parent_path + '/_iotdevice-mac.txt', 'r').read().strip()
    newpackets = rdpcap(path_in_str)
    for packet in newpackets:
        packet.device = path_in_str.split('/')[-2] # set the device name for each entry 
    packets = packets + newpackets

packets = packets.filter(lambda x: x.haslayer(TCP)) # filter only TCP packets; Not sure if we should use this or not at this point

print(packets.show())





In [ ]:
print(packets[0].show())

In [57]:
for pkt in packets:
    if hasattr(pkt, 'device'):
        x = 0
    else:
        print('No device attribute')

In [ ]:

# Read the pcap file !! Only for one file and testing purpose

dir = 'captures_IoT-Sentinel/Aria/Setup-A-1-STA.pcap'
parent_dir = os.path.dirname(dir) # get the parent directory

packets = rdpcap(dir)
mac = open(parent_dir + '/_iotdevice-mac.txt', 'r').read().strip()
print(mac)

print(len(packets))
packets = packets.filter(lambda x: x.haslayer(TCP)) # filter only TCP packets Not sure if we should use this or not
print(len(packets))



In [ ]:
# Aggregate packets into flows

flows = defaultdict(list)

for pkt in packets:

    if IP in pkt and (TCP in pkt or UDP in pkt):
       
        flow_key = (
            pkt[IP].src,
            pkt[IP].dst,
            pkt.sport,
            pkt.dport,
            pkt[IP].proto,
            pkt.device
        )
        
        flows[flow_key].append(pkt)
    

for flow, pkts in flows.items():
    print(f"Flow {flow} has {len(pkts)} packets and device is {pkts[0].device} and has number of bytes {sum([pkt.len for pkt in pkts])}")


In [ ]:
# Currently we use these features only : src_port; dst_port; proto; num_pkts; num_bytes; device = label;

input = np.zeros((1,5))
target = np.zeros((1,1))
for flow,pkts in flows.items():
    #print(flow[2], flow[3], flow[4], len(pkts), sum([pkt.len for pkt in pkts]), pkts[0].device)
    input = np.vstack([input,(flow[2], flow[3], flow[4], len(pkts), sum([pkt.len for pkt in pkts]))])
    target = np.vstack([target,pkts[0].device])

input = np.delete(input, 0, axis=0)
target = np.delete(target, 0, axis=0)
print(input) 


In [6]:
# make the labels more generic not brand names

label_mapping = {
    'Aria' : 'Device',
'D-LinkCam' : 'IP Camera',
'D-LinkDayCam' : 'IP Camera',
'D-LinkDoorSensor' : 'Sensor',
'D-LinkHomeHub' : 'Hub',
'D-LinkSensor' : 'Sensor',
'D-LinkSiren' : 'Alarm',
'D-LinkSwitch' : 'Switch',
'D-LinkWaterSensor' : 'Sensor',
'EdimaxCam1' : 'IP Camera',
'EdimaxCam2' : 'IP Camera',
'EdimaxPlug1101W' : 'Plug',
'EdimaxPlug2101W' : 'Plug',
'EdnetCam1' : 'IP Camera',
'EdnetCam2' : 'IP Camera',
'EdnetGateway' : 'Gateway',
'HomeMaticPlug' : 'Plug',
'HueBridge' : 'Hub',
'HueSwitch' : 'Switch',
'Lightify' : 'Lighting',
'MAXGateway' : 'Gateway',
'SmarterCoffee' : 'Appliance',
'TP-LinkPlugHS100' : 'Plug',
'TP-LinkPlugHS110' : 'Plug',
'WeMoInsightSwitch' : 'Switch',
'WeMoInsightSwitch2' : 'Switch',
'WeMoLink' : 'Hub',
'WeMoSwitch' : 'Switch',
'WeMoSwitch2' : 'Switch',
'Withings' : 'Device',
'iKettle2' : 'Appliance',

}


In [ ]:
# Map the old labels to the new ones
y_mapped = pd.Series(target.ravel()).map(label_mapping)

y_mapped = y_mapped.values.reshape(-1, 1)

print(len(y_mapped))
print(len(target))
print(y_mapped)


In [ ]:
data = pd.DataFrame(input, columns=['src', 'dst', 'proto', 'num_pkts', 'num_bytes'])
data['device'] = y_mapped.flatten()

label_counts = data['device'].value_counts()

# Filter out labels with fewer than 100 occurrences - to have more data points
valid_labels = label_counts[label_counts >= 0].index
valid_labels = valid_labels[valid_labels != 'Alarm']
valid_labels = valid_labels[valid_labels != 'Sensor']
filtered_data = data[data['device'].isin(valid_labels)]

print(filtered_data['device'].value_counts())

# Find the minimum number of samples across labels so that we can balance the dataset - have equal number of samples for each class
min_count = filtered_data['device'].value_counts().min()
balanced_data = filtered_data.groupby('device').sample(n=min_count, random_state=42)

X_balanced = balanced_data[['src', 'dst', 'proto', 'num_pkts', 'num_bytes']].values
y_balanced = balanced_data['device'].values



In [39]:
# Split the dataset into training and testing set - currently 80-20 split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.1, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X_train, y_train)



In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score

#y_pred = clf.predict(X_test)

probabilities = clf.predict_proba(X_test)

# Set your threshold (e.g., 0.5)
threshold = 0.5

# Get the index of the maximum probability for each prediction
max_prob_indices = np.argmax(probabilities, axis=1)

# Get the maximum probability for each prediction
max_probs = np.max(probabilities, axis=1)

# Original predictions
predictions = clf.predict(X_test)

# Add 'other' label where max probability is below the threshold
for i, prob in enumerate(max_probs):
    if prob < threshold:
        predictions[i] = 'other'


print("Classification Report:\n", classification_report(y_test, predictions))



In [ ]:
importances = clf.feature_importances_
print("Feature importances:", importances)

# Visualize the feature importances
features = ['src', 'dst', 'proto', 'num_pkts', 'num_bytes'] 
plt.bar(features, importances)
plt.xlabel('Features')
plt.ylabel('Importance')
plt.title('Feature Importances')
plt.show()

In [44]:
with open('rf_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

In [16]:
# Get the counts of the flows of each device

devices = np.unique([pkt.device for pkts in flows.values() for pkt in pkts ])
counts = {device: 0 for device in devices}

for device in devices:
    counts[device] = sum([len(pkts) for flow, pkts in flows.items() if pkts[0].device == device])



In [ ]:
# generate the input labels for the packet level one
# Currently these features src ip; dst ip; src port; dst port; packet size; ack number; flag; device is the target label


dt = np.zeros((1,6))
label = np.zeros((1,1))
for packet in packets:
    dt = np.vstack([dt, [packet[IP].src, packet[IP].dst, packet[TCP].sport, packet[TCP].dport, packet.len, packet[TCP].flags.value]])
    label = np.vstack([label,packet.device])

dt = np.delete(dt, 0, axis=0)
label = np.delete(label, 0, axis=0)
print(dt) 


In [ ]:
# Map the old labels to the new ones
label_mapped = pd.Series(label.ravel()).map(label_mapping)

label_mapped = label_mapped.values.reshape(-1, 1)

print(label_mapped)


In [ ]:
train_data = pd.DataFrame(dt, columns=['src', 'dst', 'inp_prt', 'out_port', 'num_bytes', 'flag'])
train_data['device'] = label_mapped.flatten()

label_counts = train_data['device'].value_counts()

# Filter out labels with fewer than 100 occurrences - to have more data points
valid_labels = label_counts[label_counts >= 100].index
filtered_data = train_data[train_data['device'].isin(valid_labels)]

print(filtered_data['device'].value_counts())

# Find the minimum number of samples across labels so that we can balance the dataset - have equal number of samples for each class
min_count = filtered_data['device'].value_counts().min()
balanced_data = filtered_data.groupby('device').sample(n=min_count, random_state=42)

X_balanced_data = balanced_data[['src', 'dst', 'inp_prt', 'out_port', 'num_bytes', 'flag']].values
y_balanced_data = balanced_data['device'].values


In [7]:
# Split the dataset into training and testing set - currently 80-20 split
from sklearn.model_selection import train_test_split
X_train_data, X_test_data, y_train_data, y_test_data = train_test_split(X_balanced_data, y_balanced_data, test_size=0.2, random_state=42)

In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.preprocessing import LabelEncoder

# Step 1: Graph Construction (Same as Before)
balanced_data['src'] = LabelEncoder().fit_transform(balanced_data['src'])
balanced_data['dst'] = LabelEncoder().fit_transform(balanced_data['dst'])
edge_index = torch.tensor(balanced_data[['src', 'dst']].values.T, dtype=torch.long)
node_features = torch.tensor(node_features, dtype=torch.float)
node_labels = torch.tensor(ordered_labels, dtype=torch.long)

# Step 2: Create PyTorch Geometric Data Object
graph_data = Data(x=node_features, edge_index=edge_index, y=node_labels)

# Step 3: Define GNN Model
class GNN(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GNN, self).__init__()
        self.conv1 = GCNConv(in_channels, 16)
        self.conv2 = GCNConv(16, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

# Step 4: Train the Model
model = GNN(graph_data.num_node_features, len(torch.unique(graph_data.y)))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    out = model(graph_data.x, graph_data.edge_index)
    loss = criterion(out, graph_data.y)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')
